# **Effective Agentic System Design Techniques (continued)**

In [1]:
import getpass
import os

api_key = getpass.getpass(prompt="Enter OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = api_key

In [2]:
from crewai import Agent,Task,Crew,Process
from crewai.tools import tool 
from IPython.display import display, Markdown, HTML
from crewai.llm import LLM

llm = LLM(
    model = "openai/gpt-4o-mini",
    api_key=api_key
)

## State spaces and environment modeling
In this section of Chapter 7, you read about some of the concepts of environment modeling and what Static and Dynamic environments are. These concepts ultimately tie back to how you maintain agent memory throughout an agentic workflow. So let's look at examples of the three different memory types we discussed-

Short-term memory (working memory)
Long-term memory (knowledge base)
Episodic memory (interaction history)
Most agentic frameworks supports some sort of memory management (for example CrewAI's memory management), there are other frameworks that are solely purpose built for memory management in agentic systems, such as LangMem.

### Short-term memory

In this example we use LangGraph's thread-scoped memory. This is a type of memory that lets your application remember previous interactions within a single thread or conversation. A thread organizes multiple interactions in a session, similar to the way email groups messages in a single conversation. In the following example, you will notice that we interact with the agent with two different thread_id and the agent is able to keep the memory separate for each thread rather than updating the global long-term memory.

In [3]:
from typing import TypedDict, List 
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage,BaseMessage
from langgraph.graph import StateGraph,START
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI


# Define a simple State type for our Travel Agent 
class TravelState(TypedDict):
    messages : List[BaseMessage]

# Define the travel agent response function
def generate_response(state : TravelState) -> TravelState:
    ''' Generate Response based on the conversational History'''
    system_message = SystemMessage(content="""
    You are a helpful travel agent assistant. Use the conversation history to 
    remember the user's preferences and trip details. Be specific and reference 
    their previously mentioned preferences when making recommendations.
    """)

    # Combine the system message with the existing messages
    messages = [system_message] + state["messages"]

    # Generate a response
    llm = ChatOpenAI(model="gpt-4o")
    response = llm.invoke(messages)

    # Return the updated state with the new message
    return {"messages": state["messages"] + [response]}

# Build a graph
builder = StateGraph(TravelState)
builder.add_node("generate_response", generate_response)
builder.add_edge(START, "generate_response")
builder.set_finish_point("generate_response")

# Compile the graph with a memory checkpointer
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# Function to process User messages
def chat_with_travel_agent(message: str, thread_id: str = "default"):
    """Process a user message and return the agent's response."""
    # Create the thread configuration
    config = {"configurable": {"thread_id": thread_id}}

    # Get the current state if it exists
    current_messages = []
    try:
        checkpoint = checkpointer.get_tuple(config)
        if checkpoint and "messages" in checkpoint.checkpoint.get("channel_values", {}):
            current_messages = checkpoint.checkpoint["channel_values"]["messages"]
    except:
        pass

    # Add the new message
    current_messages.append(HumanMessage(content=message))
    
    # Process through the graph
    result = graph.invoke({"messages": current_messages}, config)
    
    # Return just the last message content (the response)
    return result["messages"][-1].content


thread_id = "user_123"

# First interaction
print("User: I want to plan a trip to Japan next month.")
response = chat_with_travel_agent("I want to plan a trip to Japan next month.", thread_id)
print(f"Agent: {response}\n")
    
# Second interaction - the agent should remember Japan
print("User: I'm interested in traditional culture and my budget is $3000.")
response = chat_with_travel_agent("I'm interested in traditional culture and my budget is $3000.", thread_id)
print(f"Agent: {response}\n")

# Third interaction - test memory of previous details
print("User: What was my destination again?")
response = chat_with_travel_agent("What was my destination again?", thread_id)
print(f"Agent: {response}\n")

# New conversation thread (should not know about Japan)
new_thread = "user_456"
print("=== New Conversation ===")
print("User: What kind of budget would I need for a beach vacation?")
response = chat_with_travel_agent("What kind of budget would I need for a beach vacation?", new_thread)
print(f"Agent: {response}")

User: I want to plan a trip to Japan next month.
Agent: That sounds like an exciting trip! Since you're planning to visit Japan next month, would you like assistance with specific cities you’re interested in visiting, or activities that you enjoy? Let me know if you have any particular preferences like cultural experiences, food, nature, or if there are specific places you're eager to see!

User: I'm interested in traditional culture and my budget is $3000.
Agent: Great! With a $3000 budget, you can enjoy a rich cultural experience in Japan. Since you're interested in traditional culture, I recommend visiting Kyoto and Nara, as they are both renowned for their historical sites and traditional culture.

**Kyoto:**
1. **Kinkaku-ji (Golden Pavilion):** A stunning temple with beautiful gardens.
2. **Gion District:** Known for traditional wooden machiya houses and geisha culture.
3. **Fushimi Inari Taisha:** Famous for its thousands of red torii gates.
4. **Kyoto Tea Ceremony:** Participate

### Long-term Memory

In this example we will use LangMem to implement "memory collections" which is also known as long-term memory or knowledgebase. In this type, memories are stored as individual documents or records. For each new conversation, the memory system can decide to insert new memories to the store. In this case, a long-term memory is saved globally within a namespace named travel_preferences. Whenever we define the agentic workflow with that namespace, regardless of the thread it will refer to the globally stored preferences. This also can be useful to store information such as travel advisories, weather conditions etc. that apply globally to all travel related conversations and are perhaps not user specific.

In [6]:
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore
from langmem import create_manage_memory_tool, create_search_memory_tool
# from langchain_openai import ChatOpenAI

# Set up storage with vector embedding capabilities
store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
    }
)

# Custom tools for the travel agent
def search_flights(destination: str, dates: str, budget: str = None):
    """Search for flights based on destination, dates, and optional budget."""
    return f"Found several flight options to {destination} for {dates}. Prices range from 1200 round trip."

def search_hotels(destination: str, dates: str, preferences: str = None):
    """Search for hotels based on destination, dates, and preferences."""
    return f"Found 15 hotels in {destination} for {dates}. Options include boutique hotels, chain hotels, and vacation rentals."

def search_activities(destination: str, interests: str = None):
    """Search for activities based on destination and interests."""
    return f"Popular activities in {destination} include museums, guided tours, outdoor activities, and local cuisine experiences."

# Create a travel agent with memory capabilities
travel_agent = create_react_agent(
    "gpt-4o",
    tools=[
        # Travel-specific tools
        search_flights,
        search_hotels,
        search_activities,
        
        # Memory management tools
        create_manage_memory_tool(
            namespace=("travel_preferences",),
            instructions="""
            Proactively call this tool when you:
            1. Learn about a new user preference for travel (budget, destinations, activities, diet, etc.)
            2. Receive an explicit request to remember specific trip details
            3. Need to record important context about their upcoming travel plans
            4. Need to update incorrect or outdated information about the user's travel preferences
            """
        ),
        create_search_memory_tool(
            namespace=("travel_preferences",),
        )
    ],
    store=store,
)

# Example function to handle travel planning conversation
def travel_planning_session(user_message, user_id="user123"):
    """
    Handle a user message in the travel planning conversation.
    
    Args:
        user_message: The user's message
        user_id: Unique identifier for the user
        
    Returns:
        The agent's response
    """
    # Set up the messages with the user's input
    messages = [{"role": "user", "content": user_message}]
    
    # Invoke the agent with the configured store for memory persistence
    response = travel_agent.invoke({"messages": messages})
    
    # Return the agent's response
    return response["messages"][-1].content

# Demonstrate usage with a typical travel planning conversation

# First conversation establishing preferences
print("User: I'm planning a trip to Barcelona in June for about a week. I love architecture and food.")
response = travel_planning_session("I'm planning a trip to Barcelona in June for about a week. I love architecture and food.")
print(f"Travel Agent: {response}\n")

# Second message with budget information
print("User: My budget is around $3000 for the entire trip, and I prefer staying in boutique hotels.")
response = travel_planning_session("My budget is around $3000 for the entire trip, and I prefer staying in boutique hotels.")
print(f"Travel Agent: {response}\n")

# Asking about vegetarian restaurants (new information)
print("User: I'm vegetarian. Can you recommend some good vegetarian restaurants in Barcelona?")
response = travel_planning_session("I'm vegetarian. Can you recommend some good vegetarian restaurants in Barcelona?")
print(f"Travel Agent: {response}\n")

# Testing memory recall - should remember destination, budget, dietary preferences
print("User: What was my budget again?")
response = travel_planning_session("What was my budget again?")
print(f"Travel Agent: {response}\n")

# Testing memory recall - should remember destination, interests
print("User: Can you suggest an architecture-focused itinerary?")
response = travel_planning_session("Can you suggest an architecture-focused itinerary?")
print(f"Travel Agent: {response}")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_9840\3220509296.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  travel_agent = create_react_agent(


User: I'm planning a trip to Barcelona in June for about a week. I love architecture and food.
Travel Agent: I've found some exciting options for your trip to Barcelona in June!

### Activities:
1. **Architecture Tours**: Explore guided tours focusing on Gaudí's masterpieces like the Sagrada Familia and Park Güell.
2. **Museums**: Discover the art and history of Barcelona in its renowned museums.
3. **Local Cuisine Experiences**: Enjoy food tours that introduce you to authentic Catalan dishes.

### Hotels:
There are several options near architectural landmarks with excellent dining options:
1. **Boutique Hotels**: Unique and charming stays.
2. **Chain Hotels**: Reliable comfort with modern amenities.
3. **Vacation Rentals**: Experience local living with all the comforts of home.

Would you like more details on any specific activity or hotel?

User: My budget is around $3000 for the entire trip, and I prefer staying in boutique hotels.
Travel Agent: Got it! I’ve noted that your budget i

### Episodic memory (interaction history)

Episodic memory preserves successful interactions as learning examples that guide future behavior. Unlike short-term memory which stores facts, episodic memory captures the full context of an interaction—the situation, the thought process that led to success, and why that approach worked. These memories help the agent learn from experience, adapting its responses based on what has worked before. In the following example, we create a travel agent system with episodic memory using the langmem library. The code defines a TravelEpisode schema that captures four critical elements:

the customer's travel request
the agent's planning considerations
the specific recommendation provided, and
the positive outcome that resulted.
When the system processes a conversation where a traveler expresses concerns about Italian crowds, it stores how the agent successfully redirected them to less-crowded alternatives while still satisfying their desire for art and cuisine. This structured approach to remembering successful interactions allows the system to develop a repository of effective strategies that can be applied to similar situations in the future.

In [4]:
from pydantic import BaseModel, Field
from langmem import create_memory_manager

class TravelEpisode(BaseModel):
    """An episode captures how to handle a specific travel planning situation, including the reasoning process
    and what made it successful."""

    customer_request: str = Field(
        ..., 
        description="The travel request and relevant customer context"
    )
    considerations: str = Field(
        ...,
        description="Key travel planning considerations and reasoning process"
    )
    recommendation: str = Field(
        ...,
        description="What travel options were recommended in response"
    )
    outcome: str = Field(
        ...,
        description="How the customer responded and why the recommendation worked"
    )

manager = create_memory_manager(
    "gpt-4o",
    schemas=[TravelEpisode],
    instructions="Extract examples of successful travel planning interactions. Include the customer context, consideration process, and why the recommendation satisfied the customer.",
    enable_inserts=True,
)

# Example conversation
conversation = [
    {"role": "user", "content": "I want to visit Italy in June but I'm worried about crowds. I love art and food but hate waiting in long lines."},
    {"role": "assistant", "content": "Instead of Rome and Florence which get extremely crowded in June, consider Bologna and Ravenna. Bologna offers amazing food (it's nicknamed 'La Grassa' or 'The Fat One') and beautiful medieval architecture without the overwhelming crowds. Ravenna has stunning Byzantine mosaics and rarely has long lines. You could spend 4 days in Bologna and 2 in Ravenna with a day trip to Modena for balsamic vinegar tasting."},
    {"role": "user", "content": "That sounds perfect! I never considered those cities but they seem to match exactly what I'm looking for. How would I get between them?"},
]

episodes = manager.invoke({"messages": conversation})


In [5]:
for episode in episodes:
    print(f"{episode.content.customer_request=}")
    print(f"{episode.content.considerations=}")
    print(f"{episode.content.recommendation=}")
    print(f"{episode.content.outcome=}")

episode.content.customer_request="I want to visit Italy in June but I'm worried about crowds. I love art and food but hate waiting in long lines."
episode.content.considerations="The customer wants to avoid crowded places but enjoys art and food. Rome and Florence are popular destinations in Italy that can get extremely crowded in June. Instead, consider less crowded places that match the customer's interest in art and food."
episode.content.recommendation='Visit Bologna and Ravenna instead of Rome and Florence. Bologna offers amazing food and beautiful medieval architecture without overwhelming crowds, while Ravenna has stunning Byzantine mosaics with rarely any long lines. Suggest spending 4 days in Bologna, 2 days in Ravenna, and a day trip to Modena for balsamic vinegar tasting.'
episode.content.outcome='The customer was satisfied as the recommendations matched their interests exactly and avoided crowds. They expressed interest in visiting these cities, finding them to perfectly al

### Sequential and Parallel Processing in Agentic Workflows

Agentic workflows can execute tasks in two main ways:

1. **Parallel Processing (Hierarchical Workflow)**
2. **Sequential Processing**

The choice depends on whether agents can work independently or whether one agent needs the output of another agent.

---

### Parallel Processing (Hierarchical Workflow)

In a parallel workflow, multiple agents work on different tasks at the same time.

A **Delegator Agent** assigns tasks to multiple worker agents, and each agent completes its own task independently.

Example:

- Travel Agent delegates tasks:
  - Flight Agent → Finds flights
  - Hotel Agent → Finds hotels
  - Activity Agent → Finds activities

All agents work simultaneously, which makes the process faster.

Advantages:

- Faster execution
- Multiple tasks handled at the same time
- Useful when tasks do not depend on each other

Example:

```python
delegator_agent = Agent(
    role="Travel Coordinator",
    goal="Delegate travel planning tasks to specialized agents"
)
```

---

### Sequential Processing

In a sequential workflow, agents execute tasks **one after another**.

The output from one agent becomes the input for the next agent.

This is useful when tasks depend on previous results.

Example:

Travel planning process:

1. Flight Booking Agent:
   - Finds and confirms travel dates

2. Hotel Booking Agent:
   - Uses the confirmed dates to recommend hotels

The hotel agent cannot complete its task until the flight dates are finalized.

---

### Sequential Workflow in CrewAI

In CrewAI, sequential processing is created using:

```python
Process.sequential
```

Example:

```python
my_crew = Crew(
    agents=[agent_1, agent_2],
    tasks=[task_1, task_2],
    verbose=True,
    process=Process.sequential
)
```

Here:

- `agent_1` completes `task_1`
- The result is passed to `agent_2`
- `agent_2` completes `task_2`

---

### Difference Between Parallel and Sequential Workflows

| Feature | Parallel Workflow | Sequential Workflow |
|---|---|---|
| Execution | Agents run at the same time | Agents run one after another |
| Dependency | Tasks are independent | Tasks depend on previous results |
| Speed | Faster | Slower |
| Example | Flight search + hotel search together | Confirm flight dates before booking hotel |
| Workflow Type | Hierarchical | Sequential |

---

### When to Use Parallel Processing

Use parallel workflows when:

- Tasks can happen independently
- Speed is important
- Multiple agents can work separately

Example:

A research agent system where:

- Agent 1 researches competitors
- Agent 2 researches customers
- Agent 3 researches market trends

---

### When to Use Sequential Processing

Use sequential workflows when:

- One task requires the output of another task
- Steps must happen in a specific order
- Information flows from one agent to another

Example:

```
Planning Agent
        ↓
Flight Agent
        ↓
Hotel Agent
        ↓
Activity Agent
```

---

### Summary

Parallel workflows allow multiple agents to work together at the same time.

Sequential workflows make agents work step-by-step where each agent uses the previous agent's output.

Choosing the correct workflow improves the efficiency and accuracy of AI agent systems.